# Notebook 01 — Exploratory Data Analysis
**Run after:** `00_setup.ipynb` passes cleanly.

This notebook generates 5 analysis plots saved to `outputs/`.
No GPU needed. Pure data understanding.

| Plot | What it answers |
|---|---|
| Class distribution | How imbalanced is the data? → justifies F1 metric |
| Metadata analysis  | Age/sex/localization per class → feature insights |
| Sample images      | Do classes look similar? → justifies uncertainty |
| Pixel statistics   | Are ImageNet stats valid for this dataset? |
| Lesion duplicates  | How many duplicate lesions? → justifies GroupShuffleSplit |


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')   # run from project root so imports work

import warnings
warnings.filterwarnings('ignore')
print("Ready ✅")


Ready ✅


## Load Metadata

In [2]:
from data.dataset import load_metadata

METADATA_CSV = 'dataset/HAM10000_metadata.csv'
IMAGES_DIR   = 'dataset/images'
OUTPUTS_DIR  = 'outputs'

df = load_metadata(METADATA_CSV, IMAGES_DIR)
print(f"Loaded {len(df):,} images across {df['lesion_id'].nunique():,} unique lesions")
df.head()


STEP 1B: Loading HAM10000 Metadata
  Total records in CSV: 10015
  Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']
  Unique lesions (lesion_id): 7470
  Unique images (image_id):   10015
  All 10015 images found on disk. ✅

  Class distribution:
    akiec  (Actinic Keratosis        ):   327 (3.3%)
    bcc    (Basal Cell Carcinoma     ):   514 (5.1%)
    bkl    (Benign Keratosis         ):  1099 (11.0%)
    df     (Dermatofibroma           ):   115 (1.1%)
    mel    (Melanoma                 ):  1113 (11.1%)
    nv     (Melanocytic Nevus        ):  6705 (66.9%)
    vasc   (Vascular Lesion          ):   142 (1.4%)
Loaded 10,015 images across 7,470 unique lesions


,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset,image_path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern,dataset/images/ISIC_0027419.jpg,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern,dataset/images/ISIC_0025030.jpg,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern,dataset/images/ISIC_0026769.jpg,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern,dataset/images/ISIC_0025661.jpg,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern,dataset/images/ISIC_0031633.jpg,2


## Plot 1 — Class Distribution
**What to look for:** The nevus bar should be ~6× taller than melanoma.
This proves accuracy is the wrong metric — a model predicting "nevus" for everything
scores 67% accuracy while being clinically useless.
We use **F1-Score** instead.


In [3]:
from data.eda import plot_class_distribution
plot_class_distribution(df, OUTPUTS_DIR)

# Display inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
img = mpimg.imread(f'{OUTPUTS_DIR}/eda_1_class_distribution.png')
plt.figure(figsize=(14,5)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()



  Plotting class distribution...
    Imbalance ratio (most/least common): 58.3x
    → This confirms F1-Score + AUROC are the right metrics


## Plot 2 — Metadata Analysis
**What to look for:**
- Age violin plots: melanoma skews older patients
- Localization heatmap: back and face are most common sites
- These patterns can be used as additional features in the hybrid model


In [4]:
from data.eda import plot_metadata_analysis
plot_metadata_analysis(df, OUTPUTS_DIR)

img = mpimg.imread(f'{OUTPUTS_DIR}/eda_2_metadata_analysis.png')
plt.figure(figsize=(16,12)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()


  Plotting metadata analysis...


## Plot 3 — Sample Images Per Class
**What to look for:** The `mel` (melanoma) and `nv` (nevus) rows look
**visually very similar**. This is the core visual proof of why uncertainty
quantification is needed — even a trained dermatologist struggles with these.


In [5]:
from data.eda import plot_sample_images
plot_sample_images(df, IMAGES_DIR, OUTPUTS_DIR)

img = mpimg.imread(f'{OUTPUTS_DIR}/eda_3_sample_images.png')
plt.figure(figsize=(14,20)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()


  Plotting sample images per class...


## Plot 4 — Pixel Statistics

In [6]:
from data.eda import plot_pixel_statistics
plot_pixel_statistics(df, IMAGES_DIR, OUTPUTS_DIR, n_sample=300)

img = mpimg.imread(f'{OUTPUTS_DIR}/eda_4_pixel_statistics.png')
plt.figure(figsize=(15,4)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()


  Computing pixel statistics on 300 sampled images...
    Dataset RGB means: R=0.753 G=0.553 B=0.581
    ImageNet  RGB means: R=0.485 G=0.456 B=0.406
    Max channel deviation: 0.268 → Consider dataset-specific normalization ⚠️


## Plot 5 — Lesion Duplicates
**Why this matters:** HAM10000 has ~2,500 images that are re-photographs
of the same lesion. A naive random split puts image A of lesion_X in train
and image B of lesion_X in test — the model has *seen* the test case.
This plot proves why we must use **GroupShuffleSplit** (split by lesion_id,
not by image).


In [7]:
from data.eda import plot_lesion_duplicates
plot_lesion_duplicates(df, OUTPUTS_DIR)

img = mpimg.imread(f'{OUTPUTS_DIR}/eda_5_lesion_duplicates.png')
plt.figure(figsize=(12,4)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

print("\n→ NEXT: Open notebook 02_split_and_dataloader.ipynb")


  Plotting lesion duplicate analysis...
    Total images:           10015
    Unique lesions:         7470
    Multi-image lesions:    1956
    → GroupShuffleSplit prevents these from leaking into test set ✅

→ NEXT: Open notebook 02_split_and_dataloader.ipynb
